<a href="https://colab.research.google.com/github/Keerthi786/AI_Builder_Cup_Sustainability-Social-Impact/blob/main/Prepare_Label_DataNew.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q earthengine-api geemap rasterio

In [ ]:
import ee
import geemap
import rasterio
import numpy as np
import os
import matplotlib.pyplot as plt

In [ ]:
# Authenticate Earth Engine
ee.Authenticate()

# Initialize with your active Google Cloud Project
try:
    ee.Initialize(project='ee-surendra')
    print("Earth Engine initialized successfully!")
except Exception as e:
    print(f"Initialization failed: {e}")

Initialization failed: Caller does not have required permission to use project ee-surendra. Grant the caller the roles/serviceusage.serviceUsageConsumer role, or a custom role with the serviceusage.services.use permission, by visiting https://console.developers.google.com/iam-admin/iam?project=ee-surendra and then retry. Propagation of the new permission may take a few minutes.


### 🛠️ Alternative Solution: Create Your Own Google Cloud Project

Since you are encountering permission errors with the shared project, let's set up a new project under your own Google account where you have full ownership:

1. **Create a New Project**:
   * Open the [Google Cloud Console Project Creator](https://console.cloud.google.com/projectcreate).
   * Name your project (e.g., `my-earth-engine-project`) and click **Create**.
   * Copy your new **Project ID** from the dashboard.

2. **Enable Earth Engine API**:
   * Visit the [Earth Engine API Library Page](https://console.cloud.google.com/apis/library/earthengine.googleapis.com).
   * Ensure your new project is selected in the top-left dropdown.
   * Click **Enable**.

3. **Paste your Project ID below and run the cell**:

In [ ]:
# Replace 'YOUR-NEW-PROJECT-ID' with the Google Cloud Project ID you just created
MY_PROJECT_ID = 'first-project-1-513704'

import ee

try:
    # Re-authenticate to ensure your session binds correctly to the new project
    ee.Authenticate()
    ee.Initialize(project=MY_PROJECT_ID)
    print("🎉 Earth Engine successfully initialized with your custom project!")
except Exception as e:
    print(f"Initialization failed: {e}")
    print("Please check that Earth Engine API is enabled on your project and the Project ID is correct.")

🎉 Earth Engine successfully initialized with your custom project!


In [ ]:
# Define output folders
BASE_DIR = "/content/Builtup_CNN_Data"

TRAIN_DIR = os.path.join(BASE_DIR, "Training")
TEST_DIR = os.path.join(BASE_DIR, "Testing")

os.makedirs(TRAIN_DIR, exist_ok=True)
os.makedirs(TEST_DIR, exist_ok=True)

print("Folders created:")
print(TRAIN_DIR)
print(TEST_DIR)

Folders created:
/content/Builtup_CNN_Data/Training
/content/Builtup_CNN_Data/Testing


In [ ]:
# Define the study areas
dehradun_aoi = ee.Geometry.Rectangle([
    78.00, 30.30,
    78.05, 30.35
])

lucknow_aoi = ee.Geometry.Rectangle([
    80.95, 26.80,
    81.00, 26.85
])

In [ ]:
# Visualize the AOIs
Map = geemap.Map(basemap="Esri.WorldImagery")

Map.centerObject(dehradun_aoi, 11)

Map.addLayer(
    dehradun_aoi,
    {"color": "red"},
    "Dehradun AOI"
)

Map.addLayer(
    lucknow_aoi,
    {"color": "blue"},
    "Lucknow AOI"
)

Map

Map(center=[30.32500025013701, 78.02499999999533], controls=(WidgetControl(options=['position', 'transparent_b…

In [ ]:
Map = geemap.Map(basemap="Esri.WorldImagery")

Map.centerObject(lucknow_aoi, 11)


Map.addLayer(
    lucknow_aoi,
    {"color": "blue"},
    "Lucknow AOI"
)

Map

Map(center=[26.825000357696936, 80.9749999999938], controls=(WidgetControl(options=['position', 'transparent_b…

In [ ]:
# Sentinel-2 cloud/shadow masking
def mask_s2_scl(image):

    scl = image.select("SCL")
    mask = (
        scl.eq(4)
        .Or(scl.eq(5))
        .Or(scl.eq(6))
        .Or(scl.eq(7))
    )

    return image.updateMask(mask)

In [ ]:
# Sentinel-2 composite
S2_BANDS = [
    "B2",
    "B3",
    "B4",
    "B8"
]

def get_sentinel_composite(aoi, start_date, end_date):

    collection = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(aoi)
        .filterDate(start_date, end_date)
        .filter(
            ee.Filter.lt(
                "CLOUDY_PIXEL_PERCENTAGE",
                30
            )
        )
        .map(mask_s2_scl)
    )

    print(
        "Number of Sentinel-2 images:",
        collection.size().getInfo()
    )

    composite = (
        collection
        .median()
        .select(S2_BANDS)
        .divide(10000)
    )

    return composite

In [ ]:
# Create Dehradun image
dehradun_s2 = get_sentinel_composite(
    dehradun_aoi,
    "2020-01-01",
    "2020-12-31"
)

# Create Lucknow image
lucknow_s2 = get_sentinel_composite(
    lucknow_aoi,
    "2020-01-01",
    "2020-12-31"
)

Number of Sentinel-2 images: 88
Number of Sentinel-2 images: 84


In [ ]:
# Load ESA WorldCover 2020
worldcover = (
    ee.ImageCollection(
        "ESA/WorldCover/v100"
    )
    .first()
    .select("Map")
)

In [ ]:
# Create binary labels
dehradun_label = (
    worldcover
    .clip(dehradun_aoi)
    .eq(50)
    .rename("label")
    .uint8()
)

lucknow_label = (
    worldcover
    .clip(lucknow_aoi)
    .eq(50)
    .rename("label")
    .uint8()
)

In [ ]:
# Visualize Dehradun
Map = geemap.Map(basemap="Esri.WorldImagery")

Map.centerObject(dehradun_aoi, 11)

Map.addLayer(
    dehradun_s2,
    {
        "bands": ["B4", "B3", "B2"],
        "min": 0,
        "max": 0.3
    },
    "Dehradun Sentinel-2"
)

Map.addLayer(
    dehradun_label,
    {
        "min": 0,
        "max": 1,
        "palette": ["black", "white"]
    },
    "Dehradun Built-up Label"
)

Map

Map(center=[30.32500025013701, 78.02499999999533], controls=(WidgetControl(options=['position', 'transparent_b…

In [ ]:
# Visualize Lucknow
Map = geemap.Map(basemap="Esri.WorldImagery")

Map.centerObject(lucknow_aoi, 11)

Map.addLayer(
    lucknow_s2,
    {
        "bands": ["B4", "B3", "B2"],
        "min": 0,
        "max": 0.3
    },
    "Lucknow Sentinel-2"
)

Map.addLayer(
    lucknow_label,
    {
        "min": 0,
        "max": 1,
        "palette": ["black", "white"]
    },
    "Lucknow Built-up Label"
)

Map

Map(center=[26.825000357696936, 80.9749999999938], controls=(WidgetControl(options=['position', 'transparent_b…

In [ ]:
def export_combined_data(
    satellite_image,
    label_image,
    aoi,
    output_path
):

    # Combine satellite bands and label into one image
    combined = (
        satellite_image
        .select(["B2", "B3", "B4", "B8"])
        .addBands(
            label_image.rename("Label")
        )
    )

    # Export as ONE GeoTIFF
    geemap.ee_export_image(
        combined,
        filename=output_path,
        scale=10,
        region=aoi,
        file_per_band=False
    )

    print("Export completed:")
    print(output_path)

In [ ]:
# Export Dehradun image
dehradun_combined_path = os.path.join(
    TRAIN_DIR,
    "Dehradun_TrainingData.tif"
)

export_combined_data(
    dehradun_s2,
    dehradun_label,
    dehradun_aoi,
    dehradun_combined_path
)

Generating URL ...
Please wait ...
Data downloaded to /content/Builtup_CNN_Data/Training/Dehradun_TrainingData.tif
Export completed:
/content/Builtup_CNN_Data/Training/Dehradun_TrainingData.tif


In [ ]:
# Export Lucknow image
lucknow_combined_path = os.path.join(
    TEST_DIR,
    "Lucknow_TestingData.tif"
)

export_combined_data(
    lucknow_s2,
    lucknow_label,
    lucknow_aoi,
    lucknow_combined_path
)

Generating URL ...
Please wait ...
Data downloaded to /content/Builtup_CNN_Data/Testing/Lucknow_TestingData.tif
Export completed:
/content/Builtup_CNN_Data/Testing/Lucknow_TestingData.tif


In [ ]:
def split_image_and_label(
    combined_path,
    image_path,
    label_path
):

    with rasterio.open(combined_path) as src:

        # Read satellite bands
        satellite = src.read([1, 2, 3, 4])

        # Read label
        label = src.read(5)

        # Copy metadata
        image_meta = src.meta.copy()
        label_meta = src.meta.copy()

        # Image has 4 bands
        image_meta.update(
            count=4,
            dtype="float32"
        )

        # Label has 1 band
        label_meta.update(
            count=1,
            dtype="uint8"
        )

        # Save satellite image
        with rasterio.open(
            image_path,
            "w",
            **image_meta
        ) as dst:

            dst.write(
                satellite.astype("float32")
            )

        # Save label
        with rasterio.open(
            label_path,
            "w",
            **label_meta
        ) as dst:

            dst.write(
                label.astype("uint8"),
                1
            )

    print("Created:")
    print(image_path)
    print(label_path)

In [ ]:
# Split Dehradun
dehradun_image_path = os.path.join(
    TRAIN_DIR,
    "Dehradun_S2.tif"
)

dehradun_label_path = os.path.join(
    TRAIN_DIR,
    "Dehradun_Label.tif"
)

split_image_and_label(
    dehradun_combined_path,
    dehradun_image_path,
    dehradun_label_path
)

Created:
/content/Builtup_CNN_Data/Training/Dehradun_S2.tif
/content/Builtup_CNN_Data/Training/Dehradun_Label.tif


In [ ]:
# Split Lucknow
lucknow_image_path = os.path.join(
    TEST_DIR,
    "Lucknow_S2.tif"
)

lucknow_label_path = os.path.join(
    TEST_DIR,
    "Lucknow_Label.tif"
)

split_image_and_label(
    lucknow_combined_path,
    lucknow_image_path,
    lucknow_label_path
)

Created:
/content/Builtup_CNN_Data/Testing/Lucknow_S2.tif
/content/Builtup_CNN_Data/Testing/Lucknow_Label.tif


In [ ]:
def inspect_raster(path):

    with rasterio.open(path) as src:

        print("\nFile:", os.path.basename(path))
        print("Bands:", src.count)
        print("Width:", src.width)
        print("Height:", src.height)
        print("CRS:", src.crs)
        print("Resolution:", src.res)
        print("Transform:", src.transform)

In [ ]:
inspect_raster(dehradun_image_path)
inspect_raster(dehradun_label_path)

inspect_raster(lucknow_image_path)
inspect_raster(lucknow_label_path)


File: Dehradun_S2.tif
Bands: 4
Width: 557
Height: 557
CRS: EPSG:4326
Resolution: (8.983152841195215e-05, 8.983152841195215e-05)
Transform: | 0.00, 0.00, 78.00|
| 0.00,-0.00, 30.35|
| 0.00, 0.00, 1.00|

File: Dehradun_Label.tif
Bands: 1
Width: 557
Height: 557
CRS: EPSG:4326
Resolution: (8.983152841195215e-05, 8.983152841195215e-05)
Transform: | 0.00, 0.00, 78.00|
| 0.00,-0.00, 30.35|
| 0.00, 0.00, 1.00|

File: Lucknow_S2.tif
Bands: 4
Width: 557
Height: 557
CRS: EPSG:4326
Resolution: (8.983152841195215e-05, 8.983152841195215e-05)
Transform: | 0.00, 0.00, 80.95|
| 0.00,-0.00, 26.85|
| 0.00, 0.00, 1.00|

File: Lucknow_Label.tif
Bands: 1
Width: 557
Height: 557
CRS: EPSG:4326
Resolution: (8.983152841195215e-05, 8.983152841195215e-05)
Transform: | 0.00, 0.00, 80.95|
| 0.00,-0.00, 26.85|
| 0.00, 0.00, 1.00|


In [ ]:
def check_alignment(image_path, label_path):

    with rasterio.open(image_path) as img:
        with rasterio.open(label_path) as lbl:

            print("\nChecking:", image_path)

            print(
                "Same width:",
                img.width == lbl.width
            )

            print(
                "Same height:",
                img.height == lbl.height
            )

            print(
                "Same CRS:",
                img.crs == lbl.crs
            )

            print(
                "Same transform:",
                img.transform == lbl.transform
            )

            print(
                "Same resolution:",
                img.res == lbl.res
            )

In [ ]:
check_alignment(
    dehradun_image_path,
    dehradun_label_path
)

check_alignment(
    lucknow_image_path,
    lucknow_label_path
)


Checking: /content/Builtup_CNN_Data/Training/Dehradun_S2.tif
Same width: True
Same height: True
Same CRS: True
Same transform: True
Same resolution: True

Checking: /content/Builtup_CNN_Data/Testing/Lucknow_S2.tif
Same width: True
Same height: True
Same CRS: True
Same transform: True
Same resolution: True


In [ ]:
import shutil
from google.colab import files

# Create a ZIP file containing the prepared dataset
shutil.make_archive(
    "/content/Builtup_CNN_Data",
    "zip",
    "/content",
    "Builtup_CNN_Data"
)

print("ZIP file created successfully.")

ZIP file created successfully.


In [ ]:
files.download("/content/Builtup_CNN_Data.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>